# Travel Planner - Multi-Agent System
### Google Colab Notebook

This notebook implements a complete multi-agent travel planning system with:
- **Orchestrator Agent** - coordinates all sub-agents
- **Planner Agent** - creates day-by-day itinerary structure
- **Flight Agent** - searches for flights
- **Hotels Agent** - searches for hotels
- **Activities Agent** - finds activities
- **Budget Agent** - calculates costs
- **Output Agent** - exports to Excel
- **Blackboard Pattern** - shared memory for agent communication
- **Interactive UI** - built with ipywidgets

## Cell 1: Install Dependencies

In [ ]:
!pip install openpyxl ipywidgets -q

import asyncio
import random
import os
from datetime import date, timedelta
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum

import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

print('All dependencies installed successfully!')

## Cell 2: Data Models

In [ ]:
class PlanStatus(str, Enum):
    PENDING = 'pending'
    PLANNING = 'planning'
    FLIGHTS = 'flights'
    HOTELS = 'hotels'
    ACTIVITIES = 'activities'
    BUDGET = 'budget'
    COMPLETED = 'completed'
    FAILED = 'failed'


@dataclass
class TravelRequest:
    location: str
    start_date: date
    duration_days: int
    budget: float
    travelers: int = 1
    preferences: Optional[str] = None


@dataclass
class FlightResult:
    airline: str
    flight_number: str
    departure: str
    arrival: str
    price: float
    duration: str
    stops: int


@dataclass
class HotelResult:
    name: str
    address: str
    rating: float
    price_per_night: float
    amenities: list


@dataclass
class ActivityResult:
    name: str
    category: str
    description: str
    price: float
    duration_hours: float
    rating: float


@dataclass
class BudgetResult:
    flights_total: float
    hotels_total: float
    activities_total: float
    food_estimate: float
    transport_estimate: float
    misc_estimate: float
    total: float
    remaining: float
    currency: str = 'USD'


@dataclass
class ItineraryItem:
    time: str
    activity: str
    location: str
    cost: float
    notes: Optional[str] = None


@dataclass
class DayPlan:
    day: int
    date: str
    items: list


@dataclass
class TravelPlan:
    request: TravelRequest
    status: PlanStatus
    flights: list = field(default_factory=list)
    hotels: list = field(default_factory=list)
    activities: list = field(default_factory=list)
    budget: Optional[BudgetResult] = None
    itinerary: list = field(default_factory=list)
    excel_path: Optional[str] = None
    error: Optional[str] = None

print('Data models defined!')

## Cell 3: Blackboard (Shared Memory)

In [ ]:
class Blackboard:
    """
    Central shared memory for multi-agent communication.
    Agents publish results here; the orchestrator reads them.
    """

    def __init__(self):
        self._data = {}

    def write(self, key, value):
        self._data[key] = value

    def read(self, key, default=None):
        return self._data.get(key, default)

    def read_all(self):
        return dict(self._data)

    def clear(self):
        self._data.clear()

    # Convenience methods
    def set_request(self, request): self.write('request', request)
    def get_request(self): return self.read('request')
    def set_status(self, status): self.write('status', status)
    def get_status(self): return self.read('status')
    def set_flights(self, flights): self.write('flights', flights)
    def get_flights(self): return self.read('flights', [])
    def set_hotels(self, hotels): self.write('hotels', hotels)
    def get_hotels(self): return self.read('hotels', [])
    def set_activities(self, activities): self.write('activities', activities)
    def get_activities(self): return self.read('activities', [])
    def set_budget(self, budget): self.write('budget', budget)
    def get_budget(self): return self.read('budget')
    def set_itinerary(self, itinerary): self.write('itinerary', itinerary)
    def get_itinerary(self): return self.read('itinerary', [])
    def set_excel_path(self, path): self.write('excel_path', path)
    def get_excel_path(self): return self.read('excel_path')
    def set_error(self, error): self.write('error', error)
    def get_error(self): return self.read('error')

    def build_plan(self):
        request = self.get_request()
        if not request:
            raise ValueError('No request found on blackboard')
        return TravelPlan(
            request=request,
            status=self.get_status() or PlanStatus.PENDING,
            flights=self.get_flights(),
            hotels=self.get_hotels(),
            activities=self.get_activities(),
            budget=self.get_budget(),
            itinerary=self.get_itinerary(),
            excel_path=self.get_excel_path(),
            error=self.get_error(),
        )

print('Blackboard ready!')

## Cell 4: Planner Agent

In [ ]:
class PlannerAgent:
    """Creates the initial day-by-day itinerary skeleton."""

    def __init__(self, blackboard):
        self.blackboard = blackboard
        self.name = 'PlannerAgent'

    def run(self):
        request = self.blackboard.get_request()
        if not request:
            raise ValueError('No request found on blackboard')

        print(f'[{self.name}] Planning trip to {request.location} for {request.duration_days} days')

        itinerary = []
        start = request.start_date

        for day_num in range(1, request.duration_days + 1):
            current_date = start + timedelta(days=day_num - 1)
            items = [
                ItineraryItem(time='09:00', activity='Morning Activity', location=request.location, cost=0.0, notes='To be filled by Activities Agent'),
                ItineraryItem(time='12:30', activity='Lunch', location=request.location, cost=0.0, notes='To be filled by Budget Agent'),
                ItineraryItem(time='14:00', activity='Afternoon Activity', location=request.location, cost=0.0, notes='To be filled by Activities Agent'),
                ItineraryItem(time='19:00', activity='Dinner', location=request.location, cost=0.0, notes='To be filled by Budget Agent'),
            ]
            if day_num == request.duration_days:
                items.append(ItineraryItem(time='11:00', activity='Departure', location=request.location, cost=0.0, notes='Check-out and head to airport'))

            itinerary.append(DayPlan(day=day_num, date=current_date.isoformat(), items=items))

        self.blackboard.set_itinerary(itinerary)
        self.blackboard.set_status(PlanStatus.PLANNING)
        print(f'[{self.name}] Planning complete: {len(itinerary)} days')
        return itinerary

print('PlannerAgent ready!')

## Cell 5: Flight Agent

In [ ]:
class FlightAgent:
    """Searches for flights (mock data - replace with real API)."""

    AIRLINES = [('Emirates', 'EK'), ('Qatar Airways', 'QR'), ('Singapore Airlines', 'SQ'),
                ('Lufthansa', 'LH'), ('British Airways', 'BA'), ('Air France', 'AF'),
                ('Turkish Airlines', 'TK'), ('Japan Airlines', 'JL')]

    def __init__(self, blackboard):
        self.blackboard = blackboard
        self.name = 'FlightAgent'

    def run(self):
        request = self.blackboard.get_request()
        if not request:
            raise ValueError('No request found on blackboard')

        print(f'[{self.name}] Searching flights to {request.location}...')

        flights = []
        base_price = random.randint(300, 1200)

        for airline_name, airline_code in self.AIRLINES[:5]:
            price = round(base_price * random.uniform(0.8, 1.5), 2)
            duration_hours = random.randint(6, 16)
            duration_mins = random.randint(0, 59)
            stops = random.choices([0, 1, 2], weights=[0.3, 0.5, 0.2])[0]
            flights.append(FlightResult(
                airline=airline_name,
                flight_number=f'{airline_code} {random.randint(100, 999)}',
                departure=f'08:{random.randint(10, 59):02d}',
                arrival=f'{random.randint(10, 23):02d}:{random.randint(10, 59):02d}',
                price=price,
                duration=f'{duration_hours}h {duration_mins}m',
                stops=stops,
            ))

        flights.sort(key=lambda f: f.price)
        self.blackboard.set_flights(flights)
        self.blackboard.set_status(PlanStatus.FLIGHTS)
        print(f'[{self.name}] Found {len(flights)} flight options')
        return flights

print('FlightAgent ready!')

## Cell 6: Hotels Agent

In [ ]:
class HotelsAgent:
    """Searches for hotels (mock data - replace with real API)."""

    HOTEL_NAMES = ['Grand Palace Hotel', 'Marina Bay Resort', 'City Center Inn',
                   'Royal Orchid Hotel', 'Sunset View Resort', 'The Metropolitan',
                   'Harbor Lights Hotel', 'Mountain Lodge & Spa']

    AMENITIES_POOL = ['Free WiFi', 'Pool', 'Spa', 'Gym', 'Restaurant', 'Bar',
                      'Room Service', 'Parking', 'Airport Shuttle', 'Beach Access',
                      'Business Center', 'Concierge']

    def __init__(self, blackboard):
        self.blackboard = blackboard
        self.name = 'HotelsAgent'

    def run(self):
        request = self.blackboard.get_request()
        if not request:
            raise ValueError('No request found on blackboard')

        print(f'[{self.name}] Searching hotels in {request.location}...')

        hotels = []
        num_hotels = random.randint(4, 7)

        for i in range(num_hotels):
            name = self.HOTEL_NAMES[i % len(self.HOTEL_NAMES)]
            if i >= len(self.HOTEL_NAMES):
                name += f' {i + 1}'
            rating = round(random.uniform(3.5, 5.0), 1)
            price = round(random.uniform(50, 400) * (rating / 4.0), 2)
            amenities = random.sample(self.AMENITIES_POOL, random.randint(3, 7))
            hotels.append(HotelResult(
                name=name,
                address=f'{random.randint(1, 999)} Main St, {request.location}',
                rating=rating,
                price_per_night=price,
                amenities=amenities,
            ))

        hotels.sort(key=lambda h: h.rating, reverse=True)
        self.blackboard.set_hotels(hotels)
        self.blackboard.set_status(PlanStatus.HOTELS)
        print(f'[{self.name}] Found {len(hotels)} hotel options')
        return hotels

print('HotelsAgent ready!')

## Cell 7: Activities Agent

In [ ]:
class ActivitiesAgent:
    """Searches for activities (mock data - replace with real API)."""

    ACTIVITIES_DATA = [
        ('City Walking Tour', 'Sightseeing', 'Guided tour of the city\'s historic landmarks and hidden gems.', 25.0, 3.0, 4.7),
        ('Museum of Art', 'Culture', 'World-class art collection spanning centuries.', 20.0, 2.5, 4.5),
        ('Cooking Class', 'Food', 'Learn to cook local cuisine with a professional chef.', 65.0, 3.0, 4.8),
        ('Boat Cruise', 'Adventure', 'Scenic cruise along the coastline with dinner.', 80.0, 4.0, 4.6),
        ('Hiking Trail', 'Nature', 'Breathtaking mountain trails for all skill levels.', 15.0, 5.0, 4.4),
        ('Wine Tasting', 'Food', 'Sample local wines at a vineyard estate.', 55.0, 2.0, 4.9),
        ('Historical Landmark', 'Sightseeing', 'Iconic monument with guided historical tours.', 12.0, 1.5, 4.3),
        ('Spa & Wellness', 'Relaxation', 'Full-day spa package with massage and treatments.', 120.0, 4.0, 4.8),
        ('Market Tour', 'Culture', 'Explore local markets and street food.', 30.0, 2.5, 4.5),
        ('Water Sports', 'Adventure', 'Jet skiing, parasailing, and more.', 90.0, 3.0, 4.2),
        ('Photography Tour', 'Culture', 'Capture the best spots with a local photographer.', 45.0, 3.0, 4.6),
        ('Nightlife Experience', 'Entertainment', 'VIP access to the best clubs and bars.', 70.0, 4.0, 4.1),
    ]

    def __init__(self, blackboard):
        self.blackboard = blackboard
        self.name = 'ActivitiesAgent'

    def run(self):
        request = self.blackboard.get_request()
        if not request:
            raise ValueError('No request found on blackboard')

        print(f'[{self.name}] Searching activities in {request.location}...')

        activities = []
        num_activities = min(len(self.ACTIVITIES_DATA), random.randint(6, 10))
        selected = random.sample(self.ACTIVITIES_DATA, num_activities)

        for name, category, desc, price, duration, rating in selected:
            varied_price = round(price * random.uniform(0.9, 1.2), 2)
            activities.append(ActivityResult(
                name=name, category=category, description=desc,
                price=varied_price, duration_hours=duration, rating=rating,
            ))

        activities.sort(key=lambda a: a.rating, reverse=True)
        self.blackboard.set_activities(activities)
        self.blackboard.set_status(PlanStatus.ACTIVITIES)
        print(f'[{self.name}] Found {len(activities)} activities')
        return activities

print('ActivitiesAgent ready!')

## Cell 8: Budget Agent

In [ ]:
class BudgetAgent:
    """Calculates the total budget breakdown."""

    def __init__(self, blackboard):
        self.blackboard = blackboard
        self.name = 'BudgetAgent'

    def run(self):
        request = self.blackboard.get_request()
        if not request:
            raise ValueError('No request found on blackboard')

        flights = self.blackboard.get_flights()
        hotels = self.blackboard.get_hotels()
        activities = self.blackboard.get_activities()

        print(f'[{self.name}] Calculating budget...')

        flights_total = flights[0].price * 2 if flights else 0.0
        flights_total *= request.travelers

        hotels_total = 0.0
        if hotels:
            best_hotel = max(hotels, key=lambda h: h.rating)
            hotels_total = best_hotel.price_per_night * request.duration_days

        activities_total = 0.0
        remaining_for_activities = request.budget - flights_total - hotels_total
        for act in sorted(activities, key=lambda a: a.rating, reverse=True):
            if activities_total + act.price <= remaining_for_activities * 0.4:
                activities_total += act.price

        food_estimate = request.duration_days * request.travelers * 50.0
        transport_estimate = request.duration_days * 20.0
        misc_estimate = request.budget * 0.05

        total = flights_total + hotels_total + activities_total + food_estimate + transport_estimate + misc_estimate
        remaining = request.budget - total

        budget = BudgetResult(
            flights_total=round(flights_total, 2),
            hotels_total=round(hotels_total, 2),
            activities_total=round(activities_total, 2),
            food_estimate=round(food_estimate, 2),
            transport_estimate=round(transport_estimate, 2),
            misc_estimate=round(misc_estimate, 2),
            total=round(total, 2),
            remaining=round(remaining, 2),
        )

        self.blackboard.set_budget(budget)
        self.blackboard.set_status(PlanStatus.BUDGET)
        print(f'[{self.name}] Total: ${budget.total:.2f} (remaining: ${budget.remaining:.2f})')
        return budget

print('BudgetAgent ready!')

## Cell 9: Output Agent (Excel Export)

In [ ]:
class OutputAgent:
    """Combines all results and exports to Excel."""

    HEADER_FONT = Font(name='Calibri', bold=True, size=12, color='FFFFFF')
    HEADER_FILL = PatternFill(start_color='2F5496', end_color='2F5496', fill_type='solid')
    TITLE_FONT = Font(name='Calibri', bold=True, size=16, color='2F5496')
    SUBTITLE_FONT = Font(name='Calibri', bold=True, size=12, color='2F5496')
    NORMAL_FONT = Font(name='Calibri', size=11)
    BORDER = Border(left=Side(style='thin'), right=Side(style='thin'), top=Side(style='thin'), bottom=Side(style='thin'))

    def __init__(self, blackboard):
        self.blackboard = blackboard
        self.name = 'OutputAgent'

    def run(self):
        print(f'[{self.name}] Generating Excel output...')
        plan = self.blackboard.build_plan()
        filepath = self._export(plan)
        self.blackboard.set_excel_path(filepath)
        self.blackboard.set_status(PlanStatus.COMPLETED)
        print(f'[{self.name}] Exported to: {filepath}')
        return filepath

    def _export(self, plan):
        os.makedirs('/content/output', exist_ok=True)
        timestamp = date.today().strftime('%Y%m%d_%H%M%S')
        filename = f'travel_plan_{plan.request.location.replace(" ", "_")}_{timestamp}.xlsx'
        filepath = f'/content/output/{filename}'

        wb = Workbook()
        self._overview_sheet(wb, plan)
        self._flights_sheet(wb, plan)
        self._hotels_sheet(wb, plan)
        self._activities_sheet(wb, plan)
        self._budget_sheet(wb, plan)
        self._itinerary_sheet(wb, plan)

        if 'Sheet' in wb.sheetnames:
            del wb['Sheet']

        wb.save(filepath)
        return filepath

    def _overview_sheet(self, wb, plan):
        ws = wb.create_sheet('Overview')
        ws['A1'] = 'TRAVEL PLAN'
        ws['A1'].font = self.TITLE_FONT
        ws.merge_cells('A1:D1')
        row = 3
        for label, value in [('Destination:', plan.request.location), ('Start Date:', str(plan.request.start_date)),
                             ('Duration:', f'{plan.request.duration_days} days'), ('Travelers:', str(plan.request.travelers)),
                             ('Budget:', f'${plan.request.budget:,.2f}'), ('Status:', plan.status.value)]:
            ws.cell(row=row, column=1, value=label).font = self.SUBTITLE_FONT
            ws.cell(row=row, column=2, value=value).font = self.NORMAL_FONT
            row += 1
        row += 1
        ws.cell(row=row, column=1, value='SUMMARY').font = self.SUBTITLE_FONT
        row += 1
        for label, value in [('Flights Found:', len(plan.flights)), ('Hotels Found:', len(plan.hotels)),
                             ('Activities Found:', len(plan.activities)),
                             ('Total Cost:', f'${plan.budget.total:,.2f}' if plan.budget else 'N/A'),
                             ('Remaining:', f'${plan.budget.remaining:,.2f}' if plan.budget else 'N/A')]:
            ws.cell(row=row, column=1, value=label).font = self.NORMAL_FONT
            ws.cell(row=row, column=2, value=value).font = self.NORMAL_FONT
            row += 1
        ws.column_dimensions['A'].width = 20
        ws.column_dimensions['B'].width = 30

    def _flights_sheet(self, wb, plan):
        ws = wb.create_sheet('Flights')
        headers = ['Airline', 'Flight Number', 'Departure', 'Arrival', 'Price ($)', 'Duration', 'Stops']
        self._write_header(ws, headers)
        for i, f in enumerate(plan.flights, start=2):
            ws.cell(row=i, column=1, value=f.airline)
            ws.cell(row=i, column=2, value=f.flight_number)
            ws.cell(row=i, column=3, value=f.departure)
            ws.cell(row=i, column=4, value=f.arrival)
            ws.cell(row=i, column=5, value=f.price)
            ws.cell(row=i, column=6, value=f.duration)
            ws.cell(row=i, column=7, value=f.stops)
        self._adjust_cols(ws, headers)

    def _hotels_sheet(self, wb, plan):
        ws = wb.create_sheet('Hotels')
        headers = ['Name', 'Address', 'Rating', 'Price/Night ($)', 'Amenities']
        self._write_header(ws, headers)
        for i, h in enumerate(plan.hotels, start=2):
            ws.cell(row=i, column=1, value=h.name)
            ws.cell(row=i, column=2, value=h.address)
            ws.cell(row=i, column=3, value=h.rating)
            ws.cell(row=i, column=4, value=h.price_per_night)
            ws.cell(row=i, column=5, value=', '.join(h.amenities))
        self._adjust_cols(ws, headers)

    def _activities_sheet(self, wb, plan):
        ws = wb.create_sheet('Activities')
        headers = ['Name', 'Category', 'Description', 'Price ($)', 'Duration (hrs)', 'Rating']
        self._write_header(ws, headers)
        for i, a in enumerate(plan.activities, start=2):
            ws.cell(row=i, column=1, value=a.name)
            ws.cell(row=i, column=2, value=a.category)
            ws.cell(row=i, column=3, value=a.description)
            ws.cell(row=i, column=4, value=a.price)
            ws.cell(row=i, column=5, value=a.duration_hours)
            ws.cell(row=i, column=6, value=a.rating)
        self._adjust_cols(ws, headers)

    def _budget_sheet(self, wb, plan):
        ws = wb.create_sheet('Budget')
        if plan.budget:
            self._write_header(ws, ['Category', 'Amount ($)'])
            items = [('Flights', plan.budget.flights_total), ('Hotels', plan.budget.hotels_total),
                     ('Activities', plan.budget.activities_total), ('Food (est.)', plan.budget.food_estimate),
                     ('Transport (est.)', plan.budget.transport_estimate), ('Misc (est.)', plan.budget.misc_estimate),
                     ('', ''), ('TOTAL', plan.budget.total), ('BUDGET', plan.request.budget), ('REMAINING', plan.budget.remaining)]
            for i, (cat, amt) in enumerate(items, start=2):
                ws.cell(row=i, column=1, value=cat).font = self.NORMAL_FONT
                ws.cell(row=i, column=2, value=amt).font = self.NORMAL_FONT
                if cat in ('TOTAL', 'BUDGET', 'REMAINING'):
                    ws.cell(row=i, column=1).font = self.SUBTITLE_FONT
                    ws.cell(row=i, column=2).font = self.SUBTITLE_FONT
        self._adjust_cols(ws, ['Category', 'Amount ($)'])

    def _itinerary_sheet(self, wb, plan):
        ws = wb.create_sheet('Itinerary')
        headers = ['Day', 'Date', 'Time', 'Activity', 'Location', 'Cost ($)', 'Notes']
        self._write_header(ws, headers)
        row = 2
        for day in plan.itinerary:
            for item in day.items:
                ws.cell(row=row, column=1, value=day.day)
                ws.cell(row=row, column=2, value=day.date)
                ws.cell(row=row, column=3, value=item.time)
                ws.cell(row=row, column=4, value=item.activity)
                ws.cell(row=row, column=5, value=item.location)
                ws.cell(row=row, column=6, value=item.cost)
                ws.cell(row=row, column=7, value=item.notes or '')
                row += 1
        self._adjust_cols(ws, headers)

    def _write_header(self, ws, headers):
        for col, h in enumerate(headers, start=1):
            cell = ws.cell(row=1, column=col, value=h)
            cell.font = self.HEADER_FONT
            cell.fill = self.HEADER_FILL
            cell.alignment = Alignment(horizontal='center')
            cell.border = self.BORDER

    def _adjust_cols(self, ws, headers):
        for col, h in enumerate(headers, start=1):
            max_len = len(h)
            for row in ws.iter_rows(min_col=col, max_col=col, min_row=2, max_row=ws.max_row):
                for cell in row:
                    if cell.value:
                        max_len = max(max_len, len(str(cell.value)))
            ws.column_dimensions[get_column_letter(col)].width = min(max_len + 2, 50)

print('OutputAgent ready!')

## Cell 10: Orchestrator Agent

In [ ]:
class OrchestratorAgent:
    """Central coordinator that manages all sub-agents."""

    def __init__(self, blackboard):
        self.blackboard = blackboard
        self.name = 'OrchestratorAgent'
        self.planner = PlannerAgent(blackboard)
        self.flight_agent = FlightAgent(blackboard)
        self.hotels_agent = HotelsAgent(blackboard)
        self.activities_agent = ActivitiesAgent(blackboard)
        self.budget_agent = BudgetAgent(blackboard)
        self.output_agent = OutputAgent(blackboard)

    def run(self, request):
        print(f'\n{"="*60}')
        print(f'[{self.name}] Starting travel planning orchestration...')
        print(f'Destination: {request.location}')
        print(f'Duration: {request.duration_days} days')
        print(f'Budget: ${request.budget:,.2f}')
        print(f'{"="*60}\n')

        try:
            self.blackboard.clear()
            self.blackboard.set_request(request)
            self.blackboard.set_status(PlanStatus.PLANNING)

            # Phase 1: Planning
            print('--- Phase 1: Planning ---')
            self.planner.run()

            # Phase 2: Parallel search
            print('\n--- Phase 2: Flight + Hotels + Activities (parallel) ---')
            self.flight_agent.run()
            self.hotels_agent.run()
            self.activities_agent.run()

            # Phase 3: Budget
            print('\n--- Phase 3: Budget Calculation ---')
            self.budget_agent.run()

            # Phase 4: Output
            print('\n--- Phase 4: Output Generation ---')
            excel_path = self.output_agent.run()

            plan = self.blackboard.build_plan()
            print(f'\n{"="*60}')
            print('TRAVEL PLAN COMPLETED SUCCESSFULLY!')
            print(f'Excel file: {excel_path}')
            print(f'{"="*60}\n')
            return plan

        except Exception as e:
            error_msg = f'Travel planning failed: {str(e)}'
            print(f'\n[ERROR] {error_msg}')
            self.blackboard.set_error(error_msg)
            self.blackboard.set_status(PlanStatus.FAILED)
            return None

print('OrchestratorAgent ready!')

## Cell 11: Interactive UI

In [ ]:
class TravelPlannerUI:
    """Interactive UI using ipywidgets."""

    def __init__(self):
        self.blackboard = Blackboard()
        self.orchestrator = OrchestratorAgent(self.blackboard)
        self._build_ui()

    def _build_ui(self):
        # Input widgets
        self.location_input = widgets.Text(
            value='Paris, France',
            placeholder='Enter destination',
            description='Destination:',
            layout=widgets.Layout(width='100%')
        )
        self.date_input = widgets.DatePicker(
            description='Start Date:',
            value=date.today() + timedelta(days=30)
        )
        self.duration_input = widgets.IntSlider(
            value=7, min=1, max=30, step=1,
            description='Days:',
            style={'description_width': 'initial'}
        )
        self.budget_input = widgets.FloatText(
            value=3000.0,
            description='Budget ($):',
            style={'description_width': 'initial'}
        )
        self.travelers_input = widgets.IntSlider(
            value=2, min=1, max=10, step=1,
            description='Travelers:',
            style={'description_width': 'initial'}
        )
        self.preferences_input = widgets.Text(
            value='',
            placeholder='e.g., beach, adventure, cultural',
            description='Preferences:',
            layout=widgets.Layout(width='100%')
        )

        # Submit button
        self.submit_btn = widgets.Button(
            description='Generate Travel Plan',
            button_style='primary',
            icon='plane',
            layout=widgets.Layout(width='100%', height='48px')
        )
        self.submit_btn.on_click(self._on_submit)

        # Output area
        self.output_area = widgets.Output()
        self.status_label = widgets.HTML(value='<i>Ready to plan your trip!</i>')

        # Layout
        input_box = widgets.VBox([
            self.location_input,
            widgets.HBox([self.date_input, self.duration_input]),
            widgets.HBox([self.budget_input, self.travelers_input]),
            self.preferences_input,
            self.submit_btn,
            widgets.HTML('<hr>'),
            self.status_label,
        ], layout=widgets.Layout(padding='10px', border='1px solid #ddd', border_radius='8px'))

        self.ui = widgets.VBox([input_box, self.output_area])

    def _on_submit(self, b):
        self.submit_btn.disabled = True
        self.submit_btn.description = 'Planning...'
        self.status_label.value = '<i>Planning your trip...</i>'

        request = TravelRequest(
            location=self.location_input.value,
            start_date=self.date_input.value,
            duration_days=self.duration_input.value,
            budget=self.budget_input.value,
            travelers=self.travelers_input.value,
            preferences=self.preferences_input.value or None,
        )

        with self.output_area:
            clear_output(wait=True)
            plan = self.orchestrator.run(request)

            if plan:
                self._display_results(plan)
                self.status_label.value = '<b style="color:green">Plan generated successfully!</b>'
            else:
                self.status_label.value = '<b style="color:red">Planning failed. Check output for details.</b>'

        self.submit_btn.disabled = False
        self.submit_btn.description = 'Generate Travel Plan'

    def _display_results(self, plan):
        # Summary HTML
        summary_html = f'''
        <div style="padding:15px; background:#f0f8ff; border-radius:8px; margin:10px 0;">
            <h3 style="color:#2F5496; margin-top:0;">Travel Plan Summary</h3>
            <table style="width:100%; font-size:14px;">
                <tr><td><b>Destination:</b></td><td>{plan.request.location}</td></tr>
                <tr><td><b>Duration:</b></td><td>{plan.request.duration_days} days</td></tr>
                <tr><td><b>Travelers:</b></td><td>{plan.request.travelers}</td></tr>
                <tr><td><b>Flights Found:</b></td><td>{len(plan.flights)}</td></tr>
                <tr><td><b>Hotels Found:</b></td><td>{len(plan.hotels)}</td></tr>
                <tr><td><b>Activities Found:</b></td><td>{len(plan.activities)}</td></tr>
                <tr><td><b>Total Cost:</b></td><td>${plan.budget.total:,.2f}</td></tr>
                <tr><td><b>Remaining Budget:</b></td><td style="color:{'green' if plan.budget.remaining >= 0 else 'red'}">${plan.budget.remaining:,.2f}</td></tr>
            </table>
        </div>
        '''
        display(HTML(summary_html))

        # Flights table
        if plan.flights:
            flights_html = '<h4>Flight Options</h4><table border="1" cellpadding="5" style="border-collapse:collapse; width:100%;"><tr style="background:#2F5496; color:white;"><th>Airline</th><th>Flight</th><th>Departure</th><th>Arrival</th><th>Duration</th><th>Stops</th><th>Price</th></tr>'
            for f in plan.flights:
                flights_html += f'<tr><td>{f.airline}</td><td>{f.flight_number}</td><td>{f.departure}</td><td>{f.arrival}</td><td>{f.duration}</td><td>{f.stops}</td><td>${f.price:,.2f}</td></tr>'
            flights_html += '</table>'
            display(HTML(flights_html))

        # Hotels table
        if plan.hotels:
            hotels_html = '<h4>Hotel Options</h4><table border="1" cellpadding="5" style="border-collapse:collapse; width:100%;"><tr style="background:#2F5496; color:white;"><th>Name</th><th>Rating</th><th>Price/Night</th><th>Amenities</th></tr>'
            for h in plan.hotels:
                hotels_html += f'<tr><td>{h.name}</td><td>{"★" * int(h.rating)} {h.rating}</td><td>${h.price_per_night:.2f}</td><td>{", ".join(h.amenities)}</td></tr>'
            hotels_html += '</table>'
            display(HTML(hotels_html))

        # Activities table
        if plan.activities:
            activities_html = '<h4>Activities</h4><table border="1" cellpadding="5" style="border-collapse:collapse; width:100%;"><tr style="background:#2F5496; color:white;"><th>Name</th><th>Category</th><th>Description</th><th>Duration</th><th>Rating</th><th>Price</th></tr>'
            for a in plan.activities:
                activities_html += f'<tr><td>{a.name}</td><td>{a.category}</td><td>{a.description}</td><td>{a.duration_hours}h</td><td>{"★" * int(a.rating)} {a.rating}</td><td>${a.price:.2f}</td></tr>'
            activities_html += '</table>'
            display(HTML(activities_html))

        # Budget breakdown
        if plan.budget:
            budget_html = f'''
            <h4>Budget Breakdown</h4>
            <table border="1" cellpadding="5" style="border-collapse:collapse; width:100%;">
                <tr style="background:#2F5496; color:white;"><th>Category</th><th>Amount</th></tr>
                <tr><td>Flights</td><td>${plan.budget.flights_total:,.2f}</td></tr>
                <tr><td>Hotels</td><td>${plan.budget.hotels_total:,.2f}</td></tr>
                <tr><td>Activities</td><td>${plan.budget.activities_total:,.2f}</td></tr>
                <tr><td>Food (est.)</td><td>${plan.budget.food_estimate:,.2f}</td></tr>
                <tr><td>Transport (est.)</td><td>${plan.budget.transport_estimate:,.2f}</td></tr>
                <tr><td>Miscellaneous (est.)</td><td>${plan.budget.misc_estimate:,.2f}</td></tr>
                <tr style="font-weight:bold; background:#e8e8e8;"><td>TOTAL</td><td>${plan.budget.total:,.2f}</td></tr>
                <tr style="font-weight:bold;"><td>REMAINING</td><td style="color:{'green' if plan.budget.remaining >= 0 else 'red'}">${plan.budget.remaining:,.2f}</td></tr>
            </table>
            '''
            display(HTML(budget_html))

        # Itinerary
        if plan.itinerary:
            itinerary_html = '<h4>Day-by-Day Itinerary</h4>'
            for day in plan.itinerary:
                itinerary_html += f'<div style="margin:10px 0; padding:10px; background:#f9f9f9; border-radius:5px;"><b>Day {day.day}</b> - {day.date}<br>'
                for item in day.items:
                    itinerary_html += f'&nbsp;&nbsp;&nbsp;&nbsp;{item.time} - {item.activity}<br>'
                itinerary_html += '</div>'
            display(HTML(itinerary_html))

        # Download link
        if plan.excel_path:
            display(HTML(f'<a href="{plan.excel_path}" download style="display:inline-block; padding:10px 20px; background:#059669; color:white; text-decoration:none; border-radius:5px; margin-top:10px;">Download Excel File</a>'))

    def display(self):
        display(self.ui)


# Create and display the UI
ui = TravelPlannerUI()
ui.display()

## How to Use

1. **Run all cells** in order (Cell → Run All)
2. The interactive UI will appear at the bottom
3. Fill in your travel details:
   - **Destination** — where you want to go
   - **Start Date** — when to start
   - **Duration** — number of days
   - **Budget** — total budget in USD
   - **Travelers** — number of people
   - **Preferences** — optional preferences
4. Click **Generate Travel Plan**
5. View results: flights, hotels, activities, budget, itinerary
6. **Download Excel** file with all details

---

## Architecture

```
┌─────────────────────────────────────────────────────────┐
│                ORCHESTRATOR AGENT                        │
│              (Central Coordinator)                       │
└─────────────────────┬───────────────────────────────────┘
                      │
        ┌─────────────┼─────────────┐
        │             │             │
        ▼             ▼             ▼
┌───────────┐  ┌───────────┐  ┌───────────┐
│  PLANNER  │  │  FLIGHT   │  │  HOTELS   │
│   AGENT   │  │   AGENT   │  │   AGENT   │
└─────┬─────┘  └─────┬─────┘  └─────┬─────┘
      │              │              │
      └──────────────┼──────────────┘
                     │
              ┌──────┴──────┐
              │  BLACKBOARD  │
              │ (Shared Mem) │
              └──────┬──────┘
                     │
        ┌────────────┼────────────┐
        │            │            │
        ▼            ▼            ▼
┌───────────┐ ┌───────────┐ ┌───────────┐
│ACTIVITIES │ │  BUDGET   │ │  OUTPUT   │
│   AGENT   │ │   AGENT   │ │   AGENT   │
└───────────┘ └───────────┘ └───────────┘
```

## Agents

| Agent | Responsibility |
|-------|---------------|
| **Orchestrator** | Coordinates all agents, manages workflow |
| **Planner** | Receives user input, creates day-by-day structure |
| **Flight** | Searches for flights to destination |
| **Hotels** | Searches for hotels at destination |
| **Activities** | Finds activities and attractions |
| **Budget** | Calculates costs, ensures budget compliance |
| **Output** | Combines results, exports to Excel |

## Design Patterns

- **Blackboard Pattern** — shared memory for agent communication
- **Orchestrator Pattern** — central coordinator for agent execution